In [1]:
import torch
import torch.nn as nn

# Последовательная модель

## Первый способ создания

In [2]:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

In [3]:
model

Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
)

In [4]:
input = torch.rand([16, 784], dtype=torch.float32)

out = model(input)
out.shape

torch.Size([16, 10])

In [8]:
# Методы модели
model.state_dict()

model.state_dict()['0.weight']
model.state_dict()['0.bias']

for parameter in model.parameters():
    print(parameter)
    print(parameter.shape, end='\n\n')

# Следующие методы используется, когда нейросеть содержит слои DROPOUT и BATCHNORM
model.train() # Используется перед тренировкой модели
model.eval()  # Используется перед валидацией модели


Parameter containing:
tensor([[ 1.7265e-02, -5.0921e-05,  2.2711e-02,  ..., -1.0177e-02,
          8.7938e-03,  1.4216e-02],
        [-4.7157e-03, -2.6248e-02, -8.6999e-03,  ...,  3.1164e-02,
         -4.0021e-03, -2.2629e-02],
        [ 3.4504e-03,  1.5212e-02, -2.4243e-02,  ...,  2.3872e-02,
         -1.8566e-03,  1.8849e-02],
        ...,
        [-1.6348e-02,  3.3940e-02, -2.3835e-02,  ..., -1.4689e-02,
          2.3466e-02,  2.4117e-02],
        [ 2.5857e-02,  2.2641e-02,  4.7649e-03,  ..., -1.9515e-02,
          3.1333e-03, -2.9051e-02],
        [ 5.8638e-03,  5.8305e-03, -3.6232e-03,  ..., -1.3549e-02,
         -1.6319e-02,  2.2975e-03]], requires_grad=True)
torch.Size([128, 784])

Parameter containing:
tensor([-0.0122, -0.0119,  0.0015, -0.0114,  0.0347,  0.0011, -0.0210, -0.0089,
         0.0085,  0.0057, -0.0291, -0.0062, -0.0151, -0.0280, -0.0228, -0.0028,
        -0.0132, -0.0308,  0.0254,  0.0323,  0.0117,  0.0011,  0.0065,  0.0324,
         0.0305, -0.0182,  0.0005,  0.01

## Второй способ создания

In [9]:
model = nn.Sequential()
model.add_module('layer_1', nn.Linear(784, 128))
model.add_module('relu', nn.ReLU())
model.add_module('layer_2', nn.Linear(128, 10))

In [14]:
print(model, end='\n\n')
print(model.layer_1, end='\n\n')
print(model.relu)

Sequential(
  (layer_1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (layer_2): Linear(in_features=128, out_features=10, bias=True)
)

Linear(in_features=784, out_features=128, bias=True)

ReLU()


In [15]:
model.state_dict()

OrderedDict([('layer_1.weight',
              tensor([[-0.0188, -0.0045,  0.0157,  ...,  0.0295, -0.0252, -0.0095],
                      [-0.0208,  0.0052, -0.0008,  ...,  0.0119,  0.0065,  0.0315],
                      [-0.0143,  0.0303, -0.0192,  ...,  0.0218,  0.0342, -0.0049],
                      ...,
                      [ 0.0043,  0.0019,  0.0074,  ...,  0.0247, -0.0334, -0.0179],
                      [ 0.0039,  0.0351,  0.0101,  ...,  0.0230,  0.0355,  0.0184],
                      [ 0.0052,  0.0110,  0.0191,  ..., -0.0152, -0.0164,  0.0005]])),
             ('layer_1.bias',
              tensor([-1.3466e-02,  1.6474e-02,  1.5176e-02, -1.6631e-02, -2.3071e-03,
                       1.2386e-02,  1.5371e-02,  2.7660e-02,  5.9828e-03, -2.8690e-02,
                      -3.3546e-02, -3.1974e-02,  3.1634e-02, -2.5858e-03,  2.5516e-02,
                       2.2678e-02,  1.5747e-02, -3.5245e-02, -2.1312e-02,  1.6853e-02,
                       3.3059e-02,  7.8905e-03, -2.6486e

## Создание класса для модели нейронной сети

In [16]:
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.activation(x)
        out = self.layer_2(x)
        return out

In [17]:
# Проверяем правильность модели
model = MyModel(784, 10)

input = torch.rand([16, 784], dtype=torch.float32)
out = model(input)
print(out.shape)

torch.Size([16, 10])


## Модель с двумя входами и выходами

In [18]:
class MyModel(nn.Module):
    def __init__(self, *, input_1, input_2, output):
        super().__init__()
        self.layer_1 = nn.Linear(input_1, input_2)
        self.layer_2 = nn.Linear(input_2, output)
        self.activation = nn.ReLU()

    def forward(self, x, y):
        x = self.layer_1(x)
        x = self.activation(x + y)
        out = self.layer_2(x)
        return out, x

In [19]:
model = MyModel(input_1=784, input_2=128, output=10)

x = torch.rand([16, 784], dtype=torch.float32)
y = torch.rand([16, 128], dtype=torch.float32)

out = model(x, y)
print(f'{len(out)} \n{out[0].shape} \n{out[1].shape}')

2 
torch.Size([16, 10]) 
torch.Size([16, 128])


## Модули ModuleList и ModuleDict

In [20]:
class MyModel(nn.Module):
    def __init__(self, input, output, hidden_size=2048):
        super().__init__()
        # self.activations = nn.ModuleDict({
        #     'lrelu': nn.LeakyReLU(),
        #     'relu': nn.ReLU()
        # })
        self.layers = nn.ModuleList()
        for i in range(10):
            self.layers.add_module(f'layer_{i}', nn.Linear(input, hidden_size))
            self.layers.add_module(f'act_{i}', nn.ReLU())
            input = hidden_size
            hidden_size = int(hidden_size / 2)
        self.layers.add_module(f'output_{i}' , nn.Linear(input, output))

    def forward(self, x):
        output = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i != 0 and i % 2 == 0 and i % 4 != 0:
                output.append(x)
        output.append(x)
        return output

In [21]:
model = MyModel(784, 2)
model

MyModel(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(in_features=2048, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): ReLU()
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): ReLU()
    (10): Linear(in_features=128, out_features=64, bias=True)
    (11): ReLU()
    (12): Linear(in_features=64, out_features=32, bias=True)
    (13): ReLU()
    (14): Linear(in_features=32, out_features=16, bias=True)
    (15): ReLU()
    (16): Linear(in_features=16, out_features=8, bias=True)
    (17): ReLU()
    (18): Linear(in_features=8, out_features=4, bias=True)
    (19): ReLU()
    (20): Linear(in_features=4, out_features=2, bias=True)
  )
)

In [22]:
# Проверяем правильность построения модели.
input = torch.rand([16, 784], dtype=torch.float32)

out = model(input)

print(len(out))
print(f"out_shape_1 = {out[0].shape}")
print(f"out_shape_2 = {out[1].shape}")
print(f"out_shape_3 = {out[2].shape}")
print(f"out_shape_4 = {out[3].shape}")
print(f"out_shape_5 = {out[4].shape}")
print(f"out_shape_6 = {out[5].shape}")

6
out_shape_1 = torch.Size([16, 1024])
out_shape_2 = torch.Size([16, 256])
out_shape_3 = torch.Size([16, 64])
out_shape_4 = torch.Size([16, 16])
out_shape_5 = torch.Size([16, 4])
out_shape_6 = torch.Size([16, 2])


Добавим модуль *ModuleDict*:

In [23]:
class MyModel(nn.Module):
    def __init__(self, input, output, hidden_size=2048, choice='relu'):
        super().__init__()
        self.activations = nn.ModuleDict({
            'lrelu': nn.LeakyReLU(),
            'relu': nn.ReLU()
        })
        self.layers = nn.ModuleList()
        for i in range(10):
            self.layers.add_module(f'layer_{i}', nn.Linear(input, hidden_size))
            self.layers.add_module(f'act_{i}', self.activation)
            input = hidden_size
            hidden_size = int(hidden_size / 2)
        self.layers.add_module(f'output_{i}' , nn.Linear(input, output))

    def forward(self, x):
        output = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i != 0 and i % 2 == 0 and i % 4 != 0:
                output.append(x)
        output.append(x)
        return output

In [27]:
model = MyModel(784, 2, choice='lrelu')
model

MyModel(
  (activations): ModuleDict(
    (lrelu): LeakyReLU(negative_slope=0.01)
    (relu): ReLU()
  )
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=2048, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=2048, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): LeakyReLU(negative_slope=0.01)
    (10): Linear(in_features=128, out_features=64, bias=True)
    (11): LeakyReLU(negative_slope=0.01)
    (12): Linear(in_features=64, out_features=32, bias=True)
    (13): LeakyReLU(negative_slope=0.01)
    (14): Linear(in_features=32, out_features=16, bias=True)
    (15): LeakyReLU(negative_slope=0.01)
    (16): Linear(in_features=16, out_features=8, b

In [28]:
# Проверяем правильность построения модели.
input = torch.rand([16, 784], dtype=torch.float32)

out = model(input)

print(len(out))
print(f"out_shape_1 = {out[0].shape}")
print(f"out_shape_2 = {out[1].shape}")
print(f"out_shape_3 = {out[2].shape}")
print(f"out_shape_4 = {out[3].shape}")
print(f"out_shape_5 = {out[4].shape}")
print(f"out_shape_6 = {out[5].shape}")

6
out_shape_1 = torch.Size([16, 1024])
out_shape_2 = torch.Size([16, 256])
out_shape_3 = torch.Size([16, 64])
out_shape_4 = torch.Size([16, 16])
out_shape_5 = torch.Size([16, 4])
out_shape_6 = torch.Size([16, 2])


## Создание модели для классификации MNIST

In [29]:
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out

In [30]:
model_classification = MyModel(784, 10)

In [39]:
# Определяем функцию потерь и оптимизатор градиентного спуска
loss_classification = nn.CrossEntropyLoss()
optimizer_classification = torch.optim.Adam(model_classification.parameters(), lr=0.001)

In [40]:
# Проверяем правильность построения модели
input = torch.rand([16, 784], dtype=torch.float32)

out = model_classification(input)
print(out.shape)

torch.Size([16, 10])


## Модель для задачи регрессии

In [41]:
model_regression = MyModel(64 * 64, 2)

In [42]:
# Определяем функцию потерь и оптимизатор градиентного спуска
loss_classification = nn.MSELoss()
optimizer_classification = torch.optim.Adam(model_regression.parameters(), lr=0.001)